
# Previsão de tráfego com Prophet

Notebook inicial para importar a planilha já tratada e começar a previsão com **Prophet**.

## Antes de rodar
1. Coloque sua planilha na pasta do projeto.
2. Ajuste o nome do arquivo em `CAMINHO_ARQUIVO`.
3. Ajuste os nomes das colunas:
   - `COLUNA_DATA`
   - `COLUNA_ALVO`

> O Prophet exige um DataFrame com duas colunas principais:
> - `ds` = data
> - `y` = valor a ser previsto


In [ ]:

# Se necessário, descomente a linha abaixo para instalar o Prophet:
# !pip install prophet openpyxl scikit-learn matplotlib pandas


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:

# =========================
# AJUSTE AQUI
# =========================
CAMINHO_ARQUIVO = "dados_tratados.xlsx"   # troque para o nome correto
COLUNA_DATA = "data"                      # troque se necessário
COLUNA_ALVO = "visualizacoes"             # troque se necessário

# Se for CSV, use pd.read_csv(...)
df_raw = pd.read_excel(CAMINHO_ARQUIVO)

print("Dimensões:", df_raw.shape)
print("\nColunas disponíveis:")
print(df_raw.columns.tolist())

df_raw.head()


In [ ]:

# Conferência inicial
df_raw.info()


In [ ]:

# Preparação no formato que o Prophet exige
df = df_raw[[COLUNA_DATA, COLUNA_ALVO]].copy()
df = df.rename(columns={COLUNA_DATA: "ds", COLUNA_ALVO: "y"})

df["ds"] = pd.to_datetime(df["ds"])
df["y"] = pd.to_numeric(df["y"], errors="coerce")

# Remove faltantes, ordena e elimina datas repetidas, se houver
df = (
    df.dropna(subset=["ds", "y"])
      .sort_values("ds")
      .drop_duplicates(subset=["ds"])
      .reset_index(drop=True)
)

print(df.head())
print(df.tail())
print("\nPeríodo:", df["ds"].min(), "até", df["ds"].max())
print("Quantidade de linhas:", len(df))


In [ ]:

# Visualização da série histórica
plt.figure(figsize=(14, 5))
plt.plot(df["ds"], df["y"])
plt.title("Série histórica")
plt.xlabel("Data")
plt.ylabel("Valor")
plt.grid(True)
plt.show()



## Divisão treino/teste

Para séries temporais, **não** use divisão aleatória.  
Aqui vamos separar os últimos 30 dias para teste. Depois você pode mudar para 60, 90 etc.


In [ ]:

HORIZONTE_TESTE = 30

train = df.iloc[:-HORIZONTE_TESTE].copy()
test = df.iloc[-HORIZONTE_TESTE:].copy()

print("Treino:", train.shape)
print("Teste:", test.shape)
print("Última data de treino:", train["ds"].max())
print("Primeira data de teste:", test["ds"].min())


In [ ]:

# Modelo Prophet básico
modelo = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)

modelo.fit(train)


In [ ]:

# Cria datas futuras exatamente no tamanho do teste
future = modelo.make_future_dataframe(periods=HORIZONTE_TESTE, freq="D")

forecast = modelo.predict(future)

forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()


In [ ]:

# Gráfico geral da previsão
fig1 = modelo.plot(forecast)
plt.title("Previsão com Prophet")
plt.show()

fig2 = modelo.plot_components(forecast)
plt.show()


In [ ]:

# Junta previsão com o conjunto de teste
resultado_teste = test.merge(
    forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]],
    on="ds",
    how="left"
)

resultado_teste.head()


In [ ]:

# Métricas de avaliação
mae = mean_absolute_error(resultado_teste["y"], resultado_teste["yhat"])
rmse = mean_squared_error(resultado_teste["y"], resultado_teste["yhat"]) ** 0.5
r2 = r2_score(resultado_teste["y"], resultado_teste["yhat"])

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")


In [ ]:

# Gráfico real x previsto no período de teste
plt.figure(figsize=(14, 5))
plt.plot(train["ds"], train["y"], label="Treino")
plt.plot(test["ds"], test["y"], label="Real (teste)")
plt.plot(resultado_teste["ds"], resultado_teste["yhat"], label="Previsto")
plt.fill_between(
    resultado_teste["ds"],
    resultado_teste["yhat_lower"],
    resultado_teste["yhat_upper"],
    alpha=0.2
)
plt.title("Real x Previsto")
plt.xlabel("Data")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()



## Próximo passo

Depois de validar esse baseline, o ideal é testar:
1. **Feriados e eventos especiais** no Prophet
2. **Cross-validation temporal**
3. Ajuste de parâmetros como:
   - `changepoint_prior_scale`
   - `seasonality_prior_scale`
   - `seasonality_mode`


In [ ]:

# Exemplo opcional de previsão futura real (depois de validar o modelo)
DIAS_FUTUROS = 30

modelo_final = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.95
)
modelo_final.fit(df)

future_real = modelo_final.make_future_dataframe(periods=DIAS_FUTUROS, freq="D")
forecast_real = modelo_final.predict(future_real)

forecast_real[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(DIAS_FUTUROS)
